# 04 — Restaurant EDA

**Project:** Food Delivery Operations Analytics  
**Author:** Sitanshu Singh

Looking at restaurant performance — top restaurants by revenue, cuisine popularity, rating vs order volume, and new restaurant ramp-up.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

orders = pd.read_csv('../data/cleaned/orders_cleaned.csv', parse_dates=['order_date'])
restaurants = pd.read_csv('../data/cleaned/restaurants_cleaned.csv')

delivered = orders[orders['status'] == 'Delivered'].copy()
rest_orders = delivered.merge(restaurants, on='restaurant_id', how='left')

print(f'{len(rest_orders)} delivered orders with restaurant data')

## 1. Top 10 Restaurants by Revenue

In [ ]:
top_rests = (
    rest_orders.groupby('name')
    .agg(total_revenue=('order_value', 'sum'), total_orders=('order_id', 'count'))
    .sort_values('total_revenue', ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_rests['name'][::-1], top_rests['total_revenue'][::-1], color='#1e3a5f')
ax.set_xlabel('Total Revenue (₹)')
ax.set_title('Top 10 Restaurants by Revenue — 2023')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}k'))
plt.tight_layout()
plt.savefig('../reports/top_restaurants.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Rating vs Order Volume

In [ ]:
rest_summary = (
    rest_orders.groupby(['restaurant_id', 'name', 'avg_rating', 'cuisine_type'])
    .agg(total_orders=('order_id', 'count'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    rest_summary['avg_rating'],
    rest_summary['total_orders'],
    alpha=0.5, c='#f97316', edgecolors='none', s=40
)
ax.axvline(x=4.2, color='#ef4444', linestyle='--', label='4.2 rating threshold')
ax.set_xlabel('Restaurant Rating')
ax.set_ylabel('Total Orders (2023)')
ax.set_title('Rating vs Order Volume — the 4.2 threshold is real')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/rating_vs_orders.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Cuisine Popularity

In [ ]:
cuisine_orders = (
    rest_orders.groupby('cuisine_type')
    .agg(total_orders=('order_id', 'count'))
    .sort_values('total_orders', ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(cuisine_orders['cuisine_type'], cuisine_orders['total_orders'], color='#f97316')
ax.set_xlabel('Cuisine Type')
ax.set_ylabel('Total Orders')
ax.set_title('Order Volume by Cuisine Type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/cuisine_popularity.png', dpi=150, bbox_inches='tight')
plt.show()